# MedSAM embeddings for epithelium annotation

Computes MedSAM image embeddings for the 50 whole images chosen for epithelium annotation. This is the slow step of MedSAM: about 4.5 min per image on a laptop CPU, about a second here on a GPU. The local annotator (`app/epithelium_annotator.py`) then loads these files, so each box takes only a few seconds.

Code: [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) · `scripts/segmentation/precompute_embeddings.py`

### Before running
- **Add Input** → your dataset made from `kaggle_upload/epithelium-annotation-images.zip`.
- **Settings** → Accelerator **GPU T4**, Internet **On**.
- **Run All** (a few minutes), then download `embeddings.zip` with the link at the end.

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

matches = list(Path("/kaggle/input").rglob("annotations/epithelium/annotation_set.csv"))
assert matches, "annotation_set.csv not found under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent.parent
OUTPUT_ROOT = Path("/kaggle/working")
os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

## 2. Compute embeddings

In [ ]:
!cd {CODE_DIR} && python -m scripts.segmentation.precompute_embeddings

## 3. Download

Unzip `embeddings.zip` into the project folder on your laptop. It creates `annotations/epithelium/embeddings/`.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive(str(OUTPUT_ROOT / "embeddings"), "zip", OUTPUT_ROOT, "annotations/epithelium/embeddings")
print(archive, round(os.path.getsize(archive) / 2**20, 1), "MB,", len(list((OUTPUT_ROOT / "annotations/epithelium/embeddings").glob("*.pt"))), "files")
FileLink("embeddings.zip")